# tAilorCode - Prepare and evaluate one public task

**Status:** CPU installation and selected imports passed on October 4, 2026. GPU serving and agent evaluation have **not** run. This is not the Kaggle submission notebook and it never submits to Kaggle.

## How to use this notebook

- Work in your own Drive copy. Do not use **Run all**.
- Connecting any runtime can consume credits, even before a cell runs. Agree on consumption and session duration first.
- Use Colab **2026.04 / Python 3.12** as the verified CPU candidate. GPU availability, CUDA and memory still need checking.
- Historical outputs below are records, not proof that files or installations persist. Deleting a runtime removes its temporary files.
- Keep credentials in Colab Secrets, never in source or outputs. Each reader needs their own Kaggle account and competition access.
- Use a dedicated evaluation runtime without personal files or mounted Drive: repository code will execute in the subprocess sandbox.

See [setup and evidence](https://github.com/AraceliFradejas/tailorcode-gemma-developer-agent/blob/main/docs/colab-setup.md). Based on the organizer's [starter](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent).


## 1. Approve a bounded session

Set this flag only after agreeing on the current session's consumption and duration. It does not allocate resources, enforce billing limits or disconnect Colab. CPU preparation is not GPU/inference approval.


In [ ]:
RUNTIME_COST_APPROVED = False
if not RUNTIME_COST_APPROVED:
    raise RuntimeError("Stopped. Review and approve runtime/credit consumption before executing these cells.")


## 2. Inspect the runtime (CPU preparation)

Reports Python, platform, free disk and notebook packages. No installation or model download. A CPU-only torch here does not describe the separate environment installed later.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

import sys
import platform
import shutil
from importlib import metadata

print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.machine())
print("Disco libre (GiB):",
      round(shutil.disk_usage("/content").free / 1024**3, 1))

for name in ("kagglehub", "kaggle", "torch", "transformers", "vllm"):
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "no instalado"
    print(f"{name}: {version}")

### Recorded output - October 4, 2026

Historical output supplied by Araceli. It does not describe a newly connected runtime.

```text
Python: 3.12.13
Sistema: Linux x86_64
Disco libre (GiB): 210.5
kagglehub: 1.0.0
kaggle: 2.0.0
torch: 2.10.0+cpu
transformers: 5.0.0
vllm: no instalado
```


## 3. Authenticate privately with Colab Secrets

In the key-shaped **Secrets** sidebar, add `KAGGLE_API_TOKEN` and enable notebook access for this trusted copy. Never put its value in code or print it. KaggleHub 1.0.0 reads the secret automatically; no login widget is needed. Browser sign-in alone does not authenticate Colab.

This checks authentication without printing the account details. Do not expose secrets through outputs, environment dumps or subprocesses.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

import kagglehub

kagglehub.whoami(verbose=False)
print("Autenticación de Kaggle verificada.")

### Recorded output - October 4, 2026

Historical output supplied by Araceli. It does not describe a newly connected runtime.

```text
Autenticación de Kaggle verificada.
```


## 4. Check the pinned model configuration only

Downloads only `config.json` from model version 2, not the approximately 23.3 GB of weights. Checks access and reports quantization; it does not validate GPU serving.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

import json
from pathlib import Path
import kagglehub

handle = "google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2"

config_path = Path(
    kagglehub.model_download(handle, path="config.json")
)

config = json.loads(config_path.read_text(encoding="utf-8"))
quantization = config["quantization_config"]

print("Modelo:", handle)
print("Archivo:", config_path.name)
print("Bytes:", config_path.stat().st_size)
print("Arquitectura:", config["architectures"])
print("Cuantización:", quantization["quant_method"])
print("Estado:", quantization["quantization_status"])
print("Grupos:", json.dumps(quantization["config_groups"], indent=2))
print("Solo se ha solicitado config.json; no se ha ejecutado el modelo.")

### Recorded output - October 4, 2026

Historical output supplied by Araceli. It does not describe a newly connected runtime.

```text
Modelo: google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2
Archivo: config.json
Bytes: 18711
Arquitectura: ['Gemma4ForConditionalGeneration']
Cuantización: compressed-tensors
Estado: compressed
Grupos: {
  "group_0": {
    "format": "pack-quantized",
    "input_activations": null,
    "output_activations": null,
    "targets": [
      "Linear"
    ],
    "weights": {
      "actorder": null,
      "block_structure": null,
      "dynamic": false,
      "group_size": 32,
      "num_bits": 4,
      "observer": "memoryless_minmax",
      "observer_kwargs": {},
      "scale_dtype": null,
      "strategy": "group",
      "symmetric": true,
      "type": "int",
      "zp_dtype": null
    }
  }
}
Solo se ha solicitado config.json; no se ha ejecutado el modelo.
```


## 5. Verify public task metadata

Downloads only `tasks.jsonl` and checks the identity of `fastapi_14786`. Does not execute task code or print reference patches. Snapshots, graphs, embeddings and offline task dependencies are still separate inputs.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

import json
from pathlib import Path
import kagglehub

path = Path(kagglehub.competition_download(
    "gemma-4-developer-agent",
    path="tasks.jsonl",
))

expected = {
    "instance_id": "fastapi_14786",
    "repo": "fastapi/fastapi",
    "base_commit": "eacbce24c9d299c6a28110d9fc8ac50f53cddb08",
}

matches = []
count = 0

with path.open(encoding="utf-8") as source:
    for line in source:
        task = json.loads(line)
        count += 1
        if task.get("instance_id") == expected["instance_id"]:
            matches.append(task)

if len(matches) != 1:
    raise RuntimeError(
        f"Se esperaba una coincidencia; encontradas: {len(matches)}"
    )

for field, value in expected.items():
    if matches[0].get(field) != value:
        raise RuntimeError(
            f"La tarea no coincide con el baseline: {field}"
        )

print("Archivo:", path.name)
print("Bytes:", path.stat().st_size)
print("Registros:", count)
print("Tarea verificada:", expected["instance_id"])
print("Repositorio:", expected["repo"])
print("Commit:", expected["base_commit"])
print("No se han descargado snapshots ni pesos, ni ejecutado la tarea.")

### Recorded output - October 4, 2026

Historical output supplied by Araceli. It does not describe a newly connected runtime.

```text
Downloading to /root/.cache/kagglehub/competitions/gemma-4-developer-agent/tasks.jsonl...
Archivo: tasks.jsonl
Bytes: 1984455
Registros: 129
Tarea verificada: fastapi_14786
Repositorio: fastapi/fastapi
Commit: eacbce24c9d299c6a28110d9fc8ac50f53cddb08
No se han descargado snapshots ni pesos, ni ejecutado la tarea.
```


## 6. Prepare the project and official harness wheels

Clone the tested installer revision and recover the three small official wheels from dataset version 25. Hash checks reuse the repository manifest. No GPU packages are installed yet.

Use a fresh project folder on a new runtime. Existing folders are not overwritten; inspect them rather than deleting or rerunning blindly.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

import hashlib
import json
import platform
import shutil
import subprocess
import sys
from pathlib import Path
import kagglehub

if (platform.system(), platform.machine(), sys.version_info[:2]) != ("Linux", "x86_64", (3, 12)):
    raise RuntimeError("Use Linux x86_64 and Python 3.12.")
REPO = Path("/content/tailorcode-setup-20261004")
REVISION = "a9bfa3e54f6d57936216dcf34d15a9290e26749b"
WHEELS = Path("/content/tailorcode-harness-wheels")
VENV = Path("/content/tailorcode-gpu-venv-02")
PYTHON = VENV / "bin/python"
if REPO.exists():
    raise RuntimeError("Project folder exists. Inspect it before retrying; do not overwrite.")
subprocess.run(["git", "clone", "https://github.com/AraceliFradejas/tailorcode-gemma-developer-agent.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REVISION], check=True)
print("Project revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
WHEELS.mkdir(exist_ok=True)
manifest = json.loads((REPO / "evaluation/official-harness-wheels.json").read_text())
for entry in manifest["files"]:
    downloaded = Path(kagglehub.dataset_download("metric/gemma-4-developer-agent-wheelhouse/versions/25", path=entry["name"]))
    if hashlib.sha256(downloaded.read_bytes()).hexdigest() != entry["sha256"]:
        raise RuntimeError(f"Official wheel hash mismatch: {entry['name']}")
    shutil.copy2(downloaded, WHEELS / entry["name"])
    print("Verified:", entry["name"])


## 7. Install the isolated environment (separate preparation approval)

This downloads several GB of packages, even on CPU. Agree on the session duration before enabling installation. It does not download model weights or start inference.

Colab's notebook interpreter lacked `ensurepip`; the corrected installer uses `virtualenv` in that case. The GPU stack stays in its own environment. Use a new venv path on retries; do not overwrite a partial installation.

Progress is streamed below and also saved to a log. Retain the log and installed package list before deleting the runtime. Stop at the agreed deadline; this flag does not enforce a spending cap.


In [ ]:
INSTALL_APPROVED = False
if not globals().get("RUNTIME_COST_APPROVED", False) or not INSTALL_APPROVED:
    raise RuntimeError("Installation disabled; approve this preparation step separately.")

import importlib.util
import subprocess
import sys

if VENV.exists():
    raise RuntimeError("Environment path already exists; preserve it and choose a new path.")
if importlib.util.find_spec("ensurepip") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-setup.txt")], check=True, timeout=120)
install_log = REPO.parent / "tailorcode-install-02.log"
command = [
    sys.executable, "-u", str(REPO / "scripts/install_gpu_environment.py"),
    "--wheelhouse", str(WHEELS), "--venv", str(VENV), "--install",
]
with install_log.open("x", encoding="utf-8") as log:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
print("Installation exit code:", return_code)
if return_code != 0:
    raise RuntimeError("Installation failed. Preserve the log; do not start Gemma.")
print("Environment ready:", PYTHON)
print("No model started.")


### Recorded CPU installation outcome - October 4, 2026

The uploaded notebook records exit code **0**, `No broken requirements found.`, successful tested imports and `CUDA available: False` on CPU. The environment path was `/content/tailorcode-gpu-venv-02/bin/python`. No model started.

These are historical observations, not fresh outputs of the revised cell. The full original notebook remains with the owner; repetitive package-download output has been replaced here with this summary. This is not a lockfile or a guarantee of a future installation.


## 8. Prepare remaining inputs and review GPU consumption - pending

**Stop here until this step is prepared and approved.** Earlier checks downloaded only metadata and model configuration. The full snapshot, graph, embeddings, offline task wheels, model weights and tokenizer must exist in the evaluation runtime. Directory presence alone does not prove completeness.

Choose storage and preserve setup logs before switching runtime; do not assume CPU files or installations survive a GPU change. Do not mount personal Drive into the repository-code evaluation sandbox.

Review the assigned GPU, its memory and displayed consumption before authorizing inference. A selectable A100 is not proof of allocation or enough memory. Model files occupy about 23.3 GB; inference needs additional memory.


## 9. Configure evaluation paths

Set real input paths only after preparation. Both preflight and inference use the isolated interpreter, not the Colab notebook interpreter. The project folder and Python path must still exist in this runtime.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required")

from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess

REPO = Path("/content/tailorcode-setup-20261004")
PYTHON = Path("/content/tailorcode-gpu-venv-02/bin/python")
if not (REPO / "scripts/run_public_smoke.py").is_file() or not PYTHON.is_file():
    raise RuntimeError("Project or isolated interpreter is missing; prepare this runtime first.")
DATA_ROOT = ""
MODEL_PATH = ""
TASK_ID = "fastapi_14786"
OUTPUT = Path("/content/tailorcode-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("Evaluation interpreter:", PYTHON)


## 10. Check prerequisites without starting Gemma

A failure is a setup blocker, not an agent score. Passing basic checks does not guarantee complete files, sufficient VRAM or working CUDA/serving.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

if not RUNTIME_COST_APPROVED or not DATA_ROOT or not MODEL_PATH:
    raise RuntimeError("Runtime approval and real DATA_ROOT / MODEL_PATH paths are required")
check = subprocess.run([
    str(PYTHON), str(REPO / "scripts/check_smoke_environment.py"),
    "--data-root", DATA_ROOT, "--model-path", MODEL_PATH, "--task-id", TASK_ID,
], text=True, capture_output=True)
print(check.stdout)
print(check.stderr)
check.check_returncode()


## 11. Run one task - separately authorize inference

Keep `RUN_AGENT = False` until GPU setup and consumption are approved. The runner rechecks prerequisites. Never expose a Kaggle token to the subprocess evaluation: revoke notebook Secret access before evaluation, and do not pass credentials through environment variables or mount personal files.

The five-minute agent budget is **not a total runtime or spending cap**: startup can take up to 20 minutes, and verification adds time. This diagnostic budget differs from the baseline's 30-minute budget.

The runner attempts to stop the model server on normal completion/failure. Colab itself remains connected and can keep consuming credits; stop it after saving results. Forceful termination can bypass cleanup.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

RUN_AGENT = False
if not RUNTIME_COST_APPROVED or not RUN_AGENT:
    raise RuntimeError("Inference is disabled. Review environment and consumption before opting in.")
if not DATA_ROOT or not MODEL_PATH:
    raise RuntimeError("Set input paths first")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT.parent / (OUTPUT.name + ".log")
print("Output folder:", OUTPUT)
print("Log:", log_path)
with log_path.open("x") as log:
    run = subprocess.run([
        str(PYTHON), "-u", str(REPO / "scripts/run_public_smoke.py"),
        "--data-root", DATA_ROOT, "--model-path", MODEL_PATH,
        "--task-id", TASK_ID, "--output", str(OUTPUT), "--run-agent",
    ], stdout=log, stderr=subprocess.STDOUT)
print(log_path.read_text()[-12000:])
run.check_returncode()


## 12. Read and preserve the result

`resolved` is the public task verification outcome. `agent.patch` contains generated changes. This is not a Kaggle leaderboard score. Download the output folder and its adjacent log before deleting the runtime; saving the notebook to Drive alone does not preserve runtime files. Do not publish task reference patches or credentials.

Finally use **Runtime → Disconnect and delete runtime**, then verify there are no active sessions. This notebook never submits to Kaggle.


In [ ]:
if not globals().get("RUNTIME_COST_APPROVED", False):
    raise RuntimeError("Runtime approval is required before this step.")

report_file = OUTPUT / "run.json"
if not report_file.exists():
    print("No run report exists yet; the task has not reached the evaluation runner.")
else:
    print(report_file.read_text())
